### Practice: Parameter Efficient Fine-Tuning
In this notebook, you're gonna fine-tune large language models within limited GPU memory.

In [1]:
%pip install --quiet --upgrade peft bitsandbytes

import os
import torch
import torch.nn as nn
import torch.nn.functional as F

import transformers
from tqdm.auto import tqdm, trange
assert torch.cuda.is_available(), "you need cuda for this part"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 14.3 MB/s eta 0:00:00


In [2]:
model_name = 'unsloth/Qwen3-8B-Base-bnb-4bit'

tokenizer = transformers.AutoTokenizer.from_pretrained(model_name, device_map=device)
tokenizer.pad_token_id = tokenizer.eos_token_id

# the main model weights are loaded in 4-bit precision - but we can still tune LoRAs
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map='auto',
    low_cpu_mem_usage=True,
    offload_state_dict=True,
    load_in_4bit=True,
    torch_dtype=torch.float32,  # weights are 4-bit; layernorms and activations are fp32
)
for param in model.parameters():
    param.requires_grad=False

model.gradient_checkpointing_enable()  # only store a small subset of activations, re-compute the rest.
model.enable_input_require_grads()     # override an implementation quirk in gradient checkpoints that disables backprop unless inputs require grad
# more on gradient checkpointing: https://pytorch.org/docs/stable/checkpoint.html https://arxiv.org/abs/1604.06174

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/6.07G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

Здесь следующие ячейки не прогнаны но имеют output, потому что дальше по заданию нужно релоадить модель.

### Prompt tuning: the story of a fox (1 point)

![img](https://i.imgur.com/Ux3qQAu.png) (source: theodd1souts.fandom.com)

In [3]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

for i in range(10):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0].cpu().numpy().tolist()))


Output: A quick brown fox jumps over the lazy dog. The quick brown fox


What a blatant lie! This particular fox assures you that it didn't in fact jump over the lazy dog. No, sir! The fox was just minding its own business. __Your task is to train the model to say truth: no dog was jumped over today.__

In [4]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
outputs = model(**batch)

next_word_logits = outputs.logits[:, :-1]
true_next_tokens = batch['input_ids'][:, 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))

print("Loss:", loss)

Loss: tensor(3.5380, device='cuda:0', grad_fn=<NllLossBackward0>)


Except, we can't train the entire model - that would be 28GB gradients in float32. Instead, let's run [prompt tuning](https://arxiv.org/abs/2104.08691).

![img](https://i.imgur.com/VwNNKnb.png)


In [5]:
class WordEmbeddingsWithLearnedPrompts(nn.Module):
    """
    To perform prompt tuning, you will need to replace model's original word embeddings with a layer - THIS layer
     - that inserts trainable prompts instead of the first N token embeddings. """

    def __init__(self, word_embeddings: nn.Embedding, num_prompts: int):
        super().__init__()
        self.original_word_embeddings = word_embeddings
        self.num_prompts = num_prompts
        self.learnable_prompts = nn.Parameter(
            torch.randn(1, num_prompts, word_embeddings.embedding_dim), requires_grad=True)

    def forward(self, input_ids: torch.LongTensor):
        # input_ids shape: [batch_size, seq length]
        assert input_ids.dtype == torch.int64
        assert input_ids.shape[1] > self.num_prompts
        assert torch.all(input_ids[:, :self.num_prompts] == tokenizer.pad_token_id).item(), "don't forget to prepend several BOS tokens to input_ids"

        emb = torch.cat([
            self.learnable_prompts,
            self.original_word_embeddings(input_ids[:, self.num_prompts:])
        ], dim=-2)
        return emb

In [6]:
num_prompts = 16
test_emb_layer = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)
test_input_ids = tokenizer("a cat say on a may", return_tensors='pt')['input_ids'].to(device)

space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
test_inputs_with_prompts = torch.cat([space_for_prompts, test_input_ids], dim=1)

with torch.cuda.amp.autocast():
  test_prompt_embeddings = test_emb_layer(test_inputs_with_prompts)

assert test_prompt_embeddings.shape[:2] == test_inputs_with_prompts.shape
assert test_prompt_embeddings.shape[-1] == model.config.hidden_size
assert torch.allclose(test_prompt_embeddings[:, :num_prompts], test_emb_layer.learnable_prompts.float())
assert torch.allclose(test_prompt_embeddings[:, num_prompts:], model.model.embed_tokens(test_input_ids).float())
print("Looks legit!")

Looks legit!


/tmp/ipython-input-1152930240.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


__Now that it works,__ let's inject learnable prompts into the main model and teach it about foxes.

In [7]:
assert isinstance(model.model.embed_tokens, nn.Embedding), "you have already replaced the embedding layer. If the replacement is broken, please reload the model"

model.model.embed_tokens = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)

opt = torch.optim.Adam([model.model.embed_tokens.learnable_prompts], lr=0.01)

In [8]:
from tqdm.notebook import tqdm

the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)

outputs = model(**batch)
next_word_logits = outputs.logits[:, num_prompts : -1, :]
true_next_tokens = batch['input_ids'][:, num_prompts + 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))
print("Loss:", loss)

for step in tqdm(range(70)):
    opt.zero_grad()

    outputs = model(**batch)
    next_word_logits = outputs.logits[:, num_prompts : -1, :]
    true_next_tokens = batch['input_ids'][:, num_prompts + 1:]
    loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))
    loss.backward()
    opt.step()


print(loss)
assert loss.item() <= 0.1
print("Good job!")

Loss: tensor(3.5001, device='cuda:0', grad_fn=<NllLossBackward0>)


  0%|          | 0/70 [00:00<?, ?it/s]

tensor(0.0028, device='cuda:0', grad_fn=<NllLossBackward0>)
Good job!


In [9]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)


for i in range(15):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0, num_prompts:].cpu().numpy().tolist()))

# if you did everything right, the model will deny that the fox jumped over the lazy dog


Output: A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway


In [10]:
import gc
import torch

vars_to_delete = [
    'model', 'opt', 'real_batch', 'batch', 'outputs', 'logits', 'loss',
    'peft_config', 'next_word_logits', 'true_next_tokens', 'trainer', 'accelerator',
    'next_token'
]
for name in list(globals().keys()):
    if name in vars_to_delete and name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved : {torch.cuda.memory_reserved() / 1e9:.2f} GB")

Allocated: 5.00 GB
Reserved : 7.28 GB


### Using HuggingFace PEFT

[`peft`](https://huggingface.co/docs/peft/index) is a transformer's sister library that allows you to apply various __p__arameter __e__fficient __f__ine-__t__uning methods to pre-trained transformers. The library imlements both prompt tuning, prefix tuning, as well as several adapter-based techniques under a common interface:



In [11]:
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map='cuda',
    low_cpu_mem_usage=True,
    offload_state_dict=True,
    load_in_4bit=True,
    torch_dtype=torch.float32,  # weights are 4-bit; layernorms and activations are fp32
)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


In [12]:
import peft
assert isinstance(model.model.embed_tokens, nn.Embedding), "please reload the model"

peft_config = peft.PromptTuningConfig(task_type=peft.TaskType.CAUSAL_LM, num_virtual_tokens=16)
model = peft.get_peft_model(model, peft_config)  # note: for most peft methods, this line also modifies model in-place
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Total parameters (excluding quantization):", sum(p.numel() for p in model.parameters()))

Trainable parameters: 65536
Total parameters (excluding quantization): 4717917184


In [13]:
# Your task: optimize the PEFT-wrapped model to achieve next token prediction loss < 0.1, but this time using PEFT
# Please note: you no longer need to prepend PAD tokens, but you still need to skip :num_virtual_tokens: first logits.
# Finally, generate the sentence to make sure that the model learned the truth.

In [14]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
num_prompts = peft_config.num_virtual_tokens

opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=0.01)

real_batch = tokenizer(
    the_truth,
    return_tensors="pt",
    return_token_type_ids=False,
).to(device)

model.train()
for step in tqdm(range(70), desc="PEFT Prompt-Tuning"):
    opt.zero_grad()

    outputs = model(
        input_ids=real_batch["input_ids"],
        attention_mask=real_batch["attention_mask"],
    )
    logits = outputs.logits
    next_word_logits = logits[:, num_prompts : -1, :]
    true_next_tokens = real_batch["input_ids"][:, 1:]
    loss = F.cross_entropy(
        next_word_logits.flatten(0, 1),
        true_next_tokens.flatten(),
    )
    loss.backward()
    opt.step()

print(f"Final loss: {loss.item():.6f}")

PEFT Prompt-Tuning:   0%|          | 0/70 [00:00<?, ?it/s]

Final loss: 0.000322


### Parameter-efficient finetuning with LoRA (1 point)

When training on more serious tasks, you can use low-rank adapters based on the [LoRA paper](https://arxiv.org/pdf/2106.09685.pdf).

The core idea is to add low-rank adapters __in parallel with existing linear layers,__ like this:
<center><img src="https://i.imgur.com/6bQLNiG.png" width=240px></center>

In the original LoRA paper, the adapters were only added to attention projection matrices. However, [subsequent works](https://arxiv.org/abs/2305.14314) show that it is useful to adapt FFNs as well. But before we do any training, we need to implement the basic LoRA layer.

In [15]:
import gc
import torch

vars_to_delete = [
    'model', 'opt', 'real_batch', 'batch', 'outputs', 'logits', 'loss',
    'peft_config', 'next_word_logits', 'true_next_tokens', 'trainer', 'accelerator',
    'next_token'
]
for name in list(globals().keys()):
    if name in vars_to_delete and name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved : {torch.cuda.memory_reserved() / 1e9:.2f} GB")

Allocated: 5.00 GB
Reserved : 7.28 GB


In [16]:
# re-load the model to remove any previous PEFT tuners
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map='auto',
    low_cpu_mem_usage=True,
    offload_state_dict=True,
    load_in_4bit=True,
    torch_dtype=torch.float32,  # weights are 4-bit; layernorms and activations are fp32
).to(device)
for param in model.parameters():
    param.requires_grad=False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


In [17]:
class LoRALayer(nn.Module):
    """Wraps a linear layer with LoRA-like adapter. Wraps an existing OPT linear layer"""
    def __init__(self, module: nn.Linear, rank: int):
        super().__init__()
        self.module = module  # pre-trained (frozen) linear layer
        self.adapter_A = nn.Parameter(torch.empty(module.in_features, rank, device=module.weight.device))
        nn.init.kaiming_uniform_(self.adapter_A, a=5 ** 0.5)
        self.adapter_B = nn.Parameter(torch.zeros(rank, module.out_features, device=module.weight.device))

    def forward(self, x):
        # Apply self.module and LoRA adapter, return the sum (self.module outputs + adapter outputs)
        lora_x = x @ self.adapter_A @ self.adapter_B
        return self.module(x) + lora_x

In [18]:
# test your implementation
test_linear = nn.Linear(128, 128)
test_linear.weight.data[...] = torch.eye(128)
test_adapter = LoRALayer(test_linear, rank=8)

assert torch.allclose(test_adapter(torch.ones(1, 1, 128)), test_linear.bias + 1), "please check your forward pass"

test_adapter.adapter_A.data[...] = torch.linspace(0.1, -0.5, 128 * 8).view(128, 8)
test_adapter.adapter_B.data[...] = torch.linspace(0.5, -0.1, 128 * 8).view(8, 128)
test_linear.bias.data[...] = torch.linspace(1., -1., 128)

dummy_loss = F.mse_loss(test_adapter(torch.ones(1, 128) / 128).squeeze(), torch.linspace(-1, 1, 128))
assert torch.allclose(dummy_loss, torch.tensor(1.3711389), rtol=0, atol=1e-4)
dummy_loss.backward()
assert all(w.grad is not None for w in [test_adapter.adapter_A, test_adapter.adapter_B]), "some adapter weights have no grad"
assert torch.allclose(test_adapter.adapter_A.grad.sum(), torch.tensor(-0.60158), rtol=0, atol=1e-4), "bad grad w.r.t. A"
assert torch.allclose(test_adapter.adapter_B.grad.sum(), torch.tensor(0.9931), rtol=0, atol=1e-4), "bad grad w.r.t. B"
# note: bad grad means that your code is different from LoRA paper OR that your code is not autograd-friendly (e.g. no_grad)
del dummy_loss, test_linear, test_adapter
print("All tests passed!")

All tests passed!


### Apply LoRA to the model

The code below applies LoRA adapters on top of Q/K/V linear layers in attention blocks. You may also choose to modify other layers:
* self_attn.o_proj - attention output projection
* mlp.up_proj, mlp.gate_proj, mlp.down_proj - transformer feedforward layers
* lm_head - output LM head

__Note:__ please scroll down for the homework task

In [19]:
lora_rank = 8

for name, module in model.model.layers.named_modules():
    if 'DecoderLayer' in repr(type(module)):
        module.self_attn.q_proj = LoRALayer(module.self_attn.q_proj, rank=lora_rank).to(device)
        module.self_attn.k_proj = LoRALayer(module.self_attn.k_proj, rank=lora_rank).to(device)
        module.self_attn.v_proj = LoRALayer(module.self_attn.v_proj, rank=lora_rank).to(device)

assert sum(isinstance(module, LoRALayer) for module in model.modules()) > 0, "Did not add any LoRA layers!"

In [20]:
batch = tokenizer("This model wants to share its greatest secret:", return_tensors='pt', return_token_type_ids=False).to(device)
# test a single training step, make sure we get meaningful gradients
with torch.cuda.amp.autocast(dtype=torch.float32):
    out = model.forward(**batch)
    (out.logits.norm() / 100).backward()

for i, module in enumerate(model.modules()):
    if isinstance(module, LoRALayer):
        assert module.adapter_B.grad is not None
        assert module.adapter_B.grad.norm().item() > 0

model.zero_grad(set_to_none=True)
print("Grad check successful, well done!")

/tmp/ipython-input-2281301266.py:3: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float32):


Grad check successful, well done!


### (example) How to train your model

The example below shows how to train the LoRA adapters on a dummy dataset. You will need to run a _similar_ training task later.

__Note:__ please scroll down for the homework task

In [21]:
import gc
import torch

vars_to_delete = [
    'model', 'opt', 'real_batch', 'batch', 'outputs', 'logits', 'loss',
    'peft_config', 'next_word_logits', 'true_next_tokens', 'trainer', 'accelerator',
    'next_token', 'out', 'module', 'param', 'test_emb_layer', 'test_input_ids',
    'test_prompt_embeddings', 'test_inputs_with_prompts'
]
for name in list(globals().keys()):
    if name in vars_to_delete and name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved : {torch.cuda.memory_reserved() / 1e9:.2f} GB")

Allocated: 0.02 GB
Reserved : 0.06 GB


In [22]:
# checking if the model can learn. Change max_steps for proper training
# import datasets
# data = datasets.load_dataset("Abirate/english_quotes", split="train[:32]") # 32 lines
# data = data.map(lambda samples: tokenizer(samples['quote']), batched=True)
# model._hf_peft_config_loaded = True  # silence a warning from HF trainer

# trainer = transformers.Trainer(
#     model=model, train_dataset=data,
#     args=transformers.TrainingArguments(
#         per_device_train_batch_size=2, gradient_accumulation_steps=1,
#         # note: if you want larger batch size, increase gradient_accumulation_steps
#         warmup_steps=250, max_steps=100, learning_rate=2e-4, fp16=True,
#         logging_steps=1, output_dir='outputs', report_to=None),
#     data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
# )
# if you see cache warnings, set `model.config.use_cache = False` to silence them. Please re-enable for inference!

# trainer.train()

# NOTE: this is just an example! you do not have to wait for this progressbar to finish :)

### Final task: *actually* train the model (3 points)

Your task is to fine-tune the model to _generate python code_. Please use the above examples for inspiration. More specifically,

* __dataset:__ use [codeparrot-clean](https://huggingface.co/datasets/codeparrot/codeparrot-clean) or any other data containing python code. Since you do not need much data for this excercise, it is enough to use just shorter validation subset of `codeparrots`
* __preprocessing:__ select python code based on file extentions (.py)  (may skip in case of codeparrot - it is 100% python)
* __short lines:__ please take the first 512 characters of each line
* __adapter type:__ please use LoRA as defined above __plus at least one of:__
   - extra adapter on lm_head
   - extra adapter on MLP components (mlp.*)
   - trainable input embeddings (requires tweaking memory usage)

* __training:__ you do not have to train to convergence. If all goes well, your model should `.generate` code after 500 steps. Please use batch size of at least 4 (4 x 1 x 512 tokens) using `gradient_accumulation_steps=4`.


Note: the peft library also has LoRA implementation. However, we ask that for this assignment you show at least one complete training run with your own LoRA code.

__Alternative assignment:__ Instead of doing python code, feel free to substitute the task with any other dataset, e.g. your favorite artist or podcast, as long as it's ethical. If you choose your own task, please show examples of what your model learned - or did not learn, akin to the code examples below.

In [23]:
prompts =  ['', 'import', 'from', 'while', 'try', 'if', 'for', 'torch']  # feel free to add a few more that are not 100% assiciated with Python

# <A WHOLE LOT OF YOUR CODE>
# generate baseline samples with the selected prompts before finetuning
# please feel free to use transformers.Trainer (as above) or your custom training code
# after the training concludes, please show examples of text generated by your model. It is expected to look like Python code fragments
# print the generation examples nicely (suggestion: use pandas or HTML) for easier comparison
# note: your LoRA-enhanced model can run generation the same way as the non-trained model (above)

In [24]:
from datasets import load_dataset

raw = load_dataset(
    "ise-uiuc/Magicoder-Evol-Instruct-110K",
    split="train[:1000]",
    streaming=False
)

def extract_code(example):
    code = example["response"]
    if "```python" in code:
        code = code.split("```python")[1].split("```")[0]
    elif "```" in code:
        code = code.split("```")[1].split("```")[0]
    return {"text": code.strip()[:512]}

ds = raw.map(extract_code, remove_columns=raw.column_names)
print(f"Dataset size: {len(ds)} examples")
print(ds[0]["text"][:200] + "...")  # Preview a short Python snippet

README.md:   0%|          | 0.00/394 [00:00<?, ?B/s]

data-evol_instruct-decontaminated.jsonl:   0%|          | 0.00/255M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/111183 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset size: 1000 examples
# Establish an integer list
arr = [1, 2, 3, 4]

# Determine the length of the list
n = len(arr)

# Initialize index at 0
i = 0

# Traverse the list and output each individual element
while i < n:
    ...


In [25]:
model_name = 'unsloth/Llama-3.2-3B-bnb-4bit'

tokenizer = transformers.AutoTokenizer.from_pretrained(model_name, device_map=device)
tokenizer.pad_token_id = tokenizer.eos_token_id

model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map='auto',
    low_cpu_mem_usage=True,
    offload_state_dict=True,
    load_in_4bit=True,
    torch_dtype=torch.float32,  # weights are 4-bit; layernorms and activations are fp32
).to(device)
for param in model.parameters():
    param.requires_grad=False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

In [26]:
def generate(prompt, max_new=120, temperature=0.7):
    model.eval()
    batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(model.device)
    out = model.generate(
        **batch,
        max_new_tokens=max_new,
        do_sample=True,
        temperature=temperature,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):].strip()

In [27]:
from tqdm.notebook import tqdm

prompts = ['import', 'from', 'while', 'try', 'if', 'for', 'torch',
           'def', 'class', 'async', 'with']

before = {}
for p in tqdm(prompts):
    before[p] = generate(p, max_new=80)

  0%|          | 0/11 [00:00<?, ?it/s]

In [28]:
lora_rank = 8

for name, module in model.model.layers.named_modules():
    if 'DecoderLayer' in repr(type(module)) and not isinstance(module, LoRALayer):
        module.self_attn.q_proj = LoRALayer(module.self_attn.q_proj, rank=lora_rank).to(device)
        module.self_attn.k_proj = LoRALayer(module.self_attn.k_proj, rank=lora_rank).to(device)
        module.self_attn.v_proj = LoRALayer(module.self_attn.v_proj, rank=lora_rank).to(device)

In [29]:
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

head_lora = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=lora_rank,
    lora_alpha=16,
    target_modules=["lm_head"],
    lora_dropout=0.05,
    bias="none",
)

model = peft.get_peft_model(model, head_lora)

print("Trainable params:")
model.print_trainable_parameters()

Trainable params:
trainable params: 1,050,624 || all params: 3,217,011,712 || trainable%: 0.0327


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:693: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(


In [30]:
block_size = 512

def tokenize_fn(ex):
    out = tokenizer(
        ex["text"],
        truncation=True,
        max_length=block_size,
        padding="max_length",
        return_attention_mask=True,
    )
    out["labels"] = out["input_ids"].copy()
    return out

tokenized = ds.map(tokenize_fn, remove_columns=["text"])
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [31]:
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments,
    DataCollatorForLanguageModeling, default_data_collator
)

training_args = TrainingArguments(
    output_dir="./lora_code",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=5e-4,
    fp16=True,
    logging_steps=50,
    save_steps=500,
    eval_strategy="no",
    report_to=[],
    dataloader_num_workers=0,
    remove_unused_columns=False,
)

# Data collator that shifts labels automatically
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

trainer.train(resume_from_checkpoint=False)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
50,1.380200
100,1.309300
150,1.334900
200,1.355400
250,1.312600


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:270: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


TrainOutput(global_step=250, training_loss=1.3384786376953124, metrics={'train_runtime': 1082.4143, 'train_samples_per_second': 0.924, 'train_steps_per_second': 0.231, 'total_flos': 8672284508160000.0, 'train_loss': 1.3384786376953124, 'epoch': 1.0})

In [32]:
after = {}
for p in prompts:
    after[p] = generate(p, max_new=80)

In [33]:
# This template helps to compare generated code samples in pretty table form
# feel free to present your work in other forms

from IPython.display import HTML, display
table_template = """<table style="border:1px solid black" >
  <tr>
    <th style="text-align: center; border:1px solid black">PROMPT</th>
    <th style="text-align: center; border:1px solid black">BEFORE</th>
    <th style="text-align: center; border:1px solid black">AFTER</th>
  </tr>
{}
</table>"""

row_template = '''  <tr>
    <td style="width:20%; border:1px solid black"><pre align="left">`{}`</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
  </tr>'''

rows = []
for p in prompts:
    rows.append(row_template.format(
        p or "(empty)",
        before[p].replace("<","&lt;").replace(">","&gt;"),
        after[p].replace("<","&lt;").replace(">","&gt;"),
    ))

display(HTML(table_template.format("\n".join(rows))))

PROMPT,BEFORE,AFTER
`import`,"{Component, Input, OnInit} from '@angular/core'; import {Observable} from ""rxjs""; import {User} from ""../../model/user""; import {UserService} from ""../../service/user.service""; @Component({ selector: 'app-user', templateUrl: './user.component.html', styleUrls: ['./user.component.css'] }) export class UserComponent implements OnInit { @Input() user: User;","pandas as pd import numpy as np import datetime def main(): # Create a dataframe df = pd.read_csv('data.csv') # Print dataframe print(df) # Add a new column df['date'] = pd.to_datetime(df['date'], errors='ignore') df['day'] = df['date'].dt.dayofweek"
`from`,"flask import render_template, redirect, request, url_for, flash from flask_login import current_user, login_user, logout_user, login_required from app import app, db from app.models import User, Post from app.forms import RegistrationForm, LoginForm, PostForm # 1. route to registration page @app.route('/register', methods=['GET', 'POST']) def register","flask import Flask, request, redirect, render_template, session, flash from flask_bcrypt import Bcrypt import requests app = Flask(__name__) bcrypt = Bcrypt(app) app.secret_key ='secret' @app.route(""/"") def index(): return render_template(""index.html"") @app.route(""/register"", methods=[""POST""]) def register(): data = { ""first_name"":"
`while`,"True: x = input(""Enter a number: "") if x == ""q"": break y = int(x) if y % 2 == 0: print(""Even"") else: print(""Odd"")","(true) { let input = readline(); if (input === ""END"") break; let [x, y] = input.split("" ""); let [x1, y1] = [Number(x), Number(y)]; let [x2, y2] = [Number(x), Number(y)]; if (x1 === x2 && y1 === y2"
`try`,": from setuptools import setup except ImportError: from distutils.core import setup setup( name='pyrosetta', version='1.8.2', description='Python wrapper for ROSETTA', author='Dmitry Koptilov', author_email='dmitry.koptilov@gmail.com', url='https://github.com/d",: from.data import * from.data import * from.data import * from.data import * from.data import * from.data import * from.data import * from.data import * from.data import * from.data import * from.data import * from.data import * from.data import *
`if`,"you want to get an idea of how the 2000 presidential election would have played out in this state, the best thing to do is look at the 1996 election results. That’s because the 1996 election was the last time that the two main parties nominated candidates who were as popular as the two candidates who won the election. In that election, the Republican candidate, Bob Dole","(typeof (window) === 'undefined') { window = {}; } // Create a new window window.window1 = {}; window.window1.data = { message: 'Hello world!', name: 'John Doe' }; // Create a new window window.window2 = {}; window.window2.data = { message: 'Hello world!', name: 'Jane Doe' }; //"
`for`,3 days 3 nights. This is a private tour and you will be the only group on the tour. You will have an English speaking tour guide and a driver. All the tour activities are organized for you and your group. The hotel accommodation is in 3 star hotels. You will have a chance to explore the city of Siem Reap and visit the Angkor Temples. Please,your review The following is a list of the items of the 2007-2008 budget that are related to the following items: 1. The 2007-2008 budget includes $5 million for a new public safety building. It is the recommendation of the Public Safety Commission that the building be built in the City of West Columbia. The City of West Columbia has not indicated whether they
`torch`,".bernoulli¶ torch.bernoulli(input, dtype=None, requires_grad=False, *args, **kwargs) Returns a tensor of Bernoulli-distributed random variables. This function is only implemented for scalars. For a multi-dimensional input, use a distribution with a parameter that is a vector (e.g. torch.Distribution). Examples: >>> torch.bernoulli(torch","import torch import torch.nn as nn import torch.nn.functional as F 

У меня было мало времени пообучать модель, поэтому в результате она не сильно продвинулась за одну эпоху. Однако, в сравнении с изначальным результатом, модель сгенерировал на два питоновских примера больше, чем до этого. Также, до этого на `if` она ответила текстом, а после дообучения - питоновским кодом.

If you reach this: congratulations! you've completed everything in this practice session.

If you want to dig deeper, try to implement prompt-tuning (for bonus points!).
You can read more about prompt tuning variants in paper [1](https://arxiv.org/abs/2104.08691) or paper [2](https://arxiv.org/abs/2101.00190). Both versions can be implemented by passing trainable prompts as `model.forward(..., past_key_values=your_prompts)`.



### Read more

* How post-training quantization works: https://arxiv.org/abs/2208.07339
* An overview of running large models: https://huggingface.co/docs/accelerate/package_reference/big_modeling
* A general library for different adapter types: https://adapterhub.ml/


### [extra info] Running other models.

This notebook's code can run with other models of similar size, such as [Falcon-7B](https://huggingface.co/tiiuae/falcon-7b), [OPT-6.7B](https://huggingface.co/facebook/opt-6.7b) or [BLOOM-7.1B](https://huggingface.co/bigscience/bloom-7b1). However, they will require minor code tweaks:
1. change the model name in `AutoModelForCausalLM.from_pretrained()` __and__ `AutoTokenizer`
2. In the prompt tuning code, change `model.model.embed_tokens` to refer to the target model's word embeddings. Simply `print(model)` to navigate to them.
3. Change code to add Lora layers - specifically where you what the transformer block components, since those components now have different names.